[◀ Notebook 14: Metaprogramming](14_metaprogramming.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 16: Modules & Packages ▶](16_modules_and_packages.ipynb)

# Notebook 15: Concurrency & Asynchronous Programming

Welcome to Notebook 15 of the Bottom-Up Python Tutorial. This notebook explores Python's models for concurrent and parallel execution: Threads, Processes, and Asynchronous I/O. We will examine the mechanics of the Global Interpreter Lock (GIL), how to run CPU-bound tasks in parallel, and how to write asynchronous programs with `asyncio`.

### References:
- [Python Language Reference: Execution Model](https://docs.python.org/3/reference/executionmodel.html)
- [Python standard library: threading](https://docs.python.org/3/library/threading.html)
- [Python standard library: multiprocessing](https://docs.python.org/3/library/multiprocessing.html)
- [Python standard library: asyncio](https://docs.python.org/3/library/asyncio.html)
- [PEP 492: Coroutines with async and await syntax](https://peps.python.org/pep-0492/)

## 1. Concurrency vs. Parallelism and the GIL

Understanding the difference between concurrency and parallelism is fundamental:
- **Concurrency** is the *composition* of independently executing processes or threads. It is about dealing with a lot of things at once (typically in I/O-bound situations where tasks spend time waiting for external networks, files, or databases).
- **Parallelism** is the *simultaneous execution* of multiple tasks. It is about doing a lot of things at once (typically in CPU-bound situations that require raw calculation power).

### The Global Interpreter Lock (GIL)
CPython (the reference implementation of Python) uses a mutex lock called the **Global Interpreter Lock (GIL)**. The GIL ensures that only one OS thread executes Python bytecode at any given moment. This was introduced to simplify CPython's memory management (which uses reference counting) and make C extension integration easier.

#### Key Implications of the GIL:
1. **I/O-Bound Tasks:** Threads work exceptionally well here. When a thread performs a blocking I/O operation (like `socket.recv()` or `time.sleep()`), it releases the GIL, allowing another thread to execute.
2. **CPU-Bound Tasks:** Multi-threading does *not* speed up CPU-bound tasks in CPython. In fact, due to lock acquisition and switching overhead, running CPU-heavy tasks in multiple threads is often slower than running them sequentially.
3. **Bypassing the GIL:** To run CPU-bound code in parallel across multiple cores, Python programs must use **multiple processes** (using the `multiprocessing` module). Each process runs its own Python interpreter instance with its own private memory space and independent GIL.

## 2. Multi-threading (`threading`)

Operating system threads share the same memory namespace. In Python, you can spawn threads using the low-level `threading.Thread` or the high-level `concurrent.futures.ThreadPoolExecutor`.

In [ ]:
import threading
import time
from concurrent.futures import ThreadPoolExecutor

def io_task(name, duration):
    print(f"[Task {name}] Starting (simulating I/O)")
    time.sleep(duration)  # Releases the GIL while sleeping
    print(f"[Task {name}] Finished")
    return f"Result {name}"

# Spawning threads manually:
t1 = threading.Thread(target=io_task, args=("A", 0.1))
t2 = threading.Thread(target=io_task, args=("B", 0.1))

t1.start()
t2.start()
t1.join()
t2.join()

# Spawning threads using ThreadPoolExecutor:
with ThreadPoolExecutor(max_workers=2) as executor:
    future_c = executor.submit(io_task, "C", 0.1)
    future_d = executor.submit(io_task, "D", 0.1)
    print("C Result:", future_c.result())
    print("D Result:", future_d.result())

## 3. Multi-processing (`multiprocessing`)

To achieve true parallelism for CPU-bound tasks, we run code in separate operating system processes. Each process has its own address space. The `multiprocessing` module handles process creation, IPC (Inter-Process Communication), and provides a `ProcessPoolExecutor`.

### OS-Specific Process Start Methods:
How child processes are spawned depends on the operating system:
- **`fork`**: The default on Unix/macOS. It creates a child process by cloning the parent process exactly (copy-on-write memory). It is very fast, but child processes inherit open file descriptors and locks, which can cause deadlocks if threads are active during fork.
- **`spawn`**: The default on Windows (and available on Unix/macOS). It starts a fresh Python interpreter process from scratch. The child process only inherits necessary resources. It is slower than fork but much safer.
- **`forkserver`**: A server process is spawned at startup; subsequent processes are cloned from it via fork, avoiding inheritance of parent process threads.

### Pickling (Serialization) Restrictions:
Because child processes run in separate address spaces, data sent between processes must be serialized using Python's **`pickle`** module:
- **What can be pickled**: Basic types (`int`, `str`, `list`, etc.), functions defined at the top-level of a module, and class definitions that can be imported.
- **What cannot be pickled**: Lambda functions, nested functions, dynamically created attributes, generator objects, and open file handles or sockets. Trying to send these across processes raises a `PicklingError`.

### The Main Guard Requirement (`if __name__ == '__main__':`):
When using the `spawn` start method (mandatory on Windows), child processes import the parent module to run the worker functions. If you do not wrap your main process code in `if __name__ == '__main__':`, the child processes will execute the spawning code recursively, leading to infinite loops and a `RuntimeError`.

In [ ]:
import time
import os
from concurrent.futures import ProcessPoolExecutor

# In Jupyter on Windows, multiprocessing child processes cannot access 
# functions defined dynamically in the notebook cells. To work around this,
# we write the function to a separate helper file and import it.
with open("multiprocess_helper.py", "w") as f:
    f.write("""
def cpu_heavy_square_sum(n):
    total = 0
    for i in range(n):
        total += i * i
    return total
""")

from multiprocess_helper import cpu_heavy_square_sum

if __name__ == "__main__":
    numbers = [3_000_000, 3_000_001, 3_000_002]
    start = time.time()
    
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(cpu_heavy_square_sum, numbers))
        
    end = time.time()
    print(f"Results: {results}")
    print(f"Time taken with multiprocessing: {end - start:.4f} seconds")
    
    # Clean up the helper file
    try:
        os.remove("multiprocess_helper.py")
    except OSError:
        pass

## 4. Asynchronous Programming (`asyncio`)

Asynchronous programming is a single-threaded concurrency model that uses **cooperative multitasking**. Instead of the operating system preemptively scheduling threads, the Python application controls task switching explicitly through an **Event Loop**.

- **Coroutines:** Declared using `async def`. Inside a coroutine, the `await` keyword yields control back to the event loop, allowing other coroutines to execute while waiting for I/O.
- **Tasks:** Scheduled coroutines that run concurrently.

### Async Context Managers:
An async context manager is entered via `async with`. It implements the asynchronous context manager protocol:
- **`__aenter__(self)`**: A coroutine called at the start of the block. Its returned value is bound to the target variable in the `as` clause.
- **`__aexit__(self, exc_type, exc_val, exc_tb)`**: A coroutine called at the end of the block. Like `__exit__`, it receives any exception details, and returning a truthy value suppresses the exception.

### Asynchronous Exception Handling & Gathering:
- **`asyncio.gather`**: Schedules multiple tasks concurrently. By default (`return_exceptions=False`), if one task raises an exception, it propagates immediately, but other tasks are *not* cancelled and continue running in the background. Setting `return_exceptions=True` causes exceptions to be returned as items in the results list alongside successful values.
- **Task Groups (Python 3.11+)**: Entered via `async with asyncio.TaskGroup() as tg:`. If a task spawned in a TaskGroup raises an exception, all other active tasks in the group are immediately cancelled. When the block exits, any raised exceptions are packaged into a single **`ExceptionGroup`**.

In [ ]:
import asyncio
import time

# Simple Mock Async Context Manager
class AsyncResource:
    async def __aenter__(self):
        print("-> Async Enter resource context")
        await asyncio.sleep(0.05)
        return self

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        print("-> Async Exit resource context")
        await asyncio.sleep(0.05)
        return True

async def fetch_data(task_id, delay):
    print(f"[Task {task_id}] Fetching...")
    async with AsyncResource():
        await asyncio.sleep(delay)  # Yields control back to the loop
    print(f"[Task {task_id}] Done")
    return f"Data {task_id}"

async def main():
    start = time.time()
    
    # 1. Gathering concurrent tasks
    print("--- Running gather concurrency ---")
    results = await asyncio.gather(
        fetch_data(1, 0.1),
        fetch_data(2, 0.05)
    )
    print("Gather results:", results)
    
    # 2. Concurrency using Task Groups (Python 3.11+)
    print("\n--- Running TaskGroup concurrency ---")
    async with asyncio.TaskGroup() as tg:
        t1 = tg.create_task(fetch_data(3, 0.08))
        t2 = tg.create_task(fetch_data(4, 0.04))
    print("TaskGroup results:", [t1.result(), t2.result()])
    
    print(f"\nTotal async time: {time.time() - start:.4f}s")

await main()


---

## 5. Coding Challenges

Complete the three challenges below. Assert checks are provided at the end of each challenge to verify your implementation.

### Challenge 1: Comparative Synchronous vs. Multi-threaded vs. Async Fetcher

Implement three versions of a simulated URL downloader that downloads a list of URLs. Each URL download is simulated by sleeping for `0.1` seconds. 
- `fetch_sync(urls)` should run sequentially using `time.sleep`.
- `fetch_threaded(urls)` should run in parallel using `ThreadPoolExecutor` and `time.sleep`.
- `fetch_async(urls)` should run concurrently using `asyncio.gather` and `asyncio.sleep`.

In [ ]:
import time
import asyncio
from concurrent.futures import ThreadPoolExecutor

def fetch_sync(urls):
    # TODO: Implement synchronous fetching sequentially
    pass

def fetch_threaded(urls):
    # TODO: Implement multi-threaded fetching using ThreadPoolExecutor
    pass

async def fetch_async(urls):
    # TODO: Implement asynchronous fetching using asyncio.gather
    pass

In [ ]:
# Solution Code for Challenge 1 (if you need a reference, you can uncomment and run it)
# def fetch_sync(urls):
#     for url in urls:
#         time.sleep(0.1)
# 
# def fetch_threaded(urls):
#     with ThreadPoolExecutor(max_workers=len(urls)) as exec:
#         exec.map(lambda url: time.sleep(0.1), urls)
# 
# async def fetch_async(urls):
#     async def download(url):
#         await asyncio.sleep(0.1)
#     await asyncio.gather(*(download(url) for url in urls))

In [ ]:
# Challenge 1 Verification Test
urls = [f"http://example.com/{i}" for i in range(4)]

# 1. Test Sync
t_start = time.time()
fetch_sync(urls)
d_sync = time.time() - t_start

# 2. Test Threaded
t_start = time.time()
fetch_threaded(urls)
d_threaded = time.time() - t_start

# 3. Test Async
t_start = time.time()
await fetch_async(urls)
d_async = time.time() - t_start

print(f"Sync duration: {d_sync:.4f}s")
print(f"Threaded duration: {d_threaded:.4f}s")
print(f"Async duration: {d_async:.4f}s")

assert d_sync >= 0.4, f"Sync should take at least 0.4s, took {d_sync:.4f}s"
assert d_threaded < 0.25, f"Threaded should be concurrent (< 0.25s), took {d_threaded:.4f}s"
assert d_async < 0.25, f"Async should be concurrent (< 0.25s), took {d_async:.4f}s"
print("Challenge 1 Passed Successfully!")

### Challenge 2: CPU-bound Parallelizer

Write a function `parallel_heavy_computation(data_list)` that calculates the sum of squares of numbers from `0` to `n` for each integer `n` in `data_list` using `ProcessPoolExecutor` to run calculations in parallel processes, bypassing the GIL.

In [ ]:
from concurrent.futures import ProcessPoolExecutor

def heavy_calculation(n):
    return sum(i * i for i in range(n))

def parallel_heavy_computation(data_list):
    # TODO: Run heavy_calculation(n) for each n in data_list in parallel using ProcessPoolExecutor
    pass

In [ ]:
# Solution Code for Challenge 2
# def parallel_heavy_computation(data_list):
#     with ProcessPoolExecutor() as executor:
#         return list(executor.map(heavy_calculation, data_list))

In [ ]:
# Challenge 2 Verification Test
test_data = [100000, 150000, 200000]
results = parallel_heavy_computation(test_data)

expected = [
    sum(i * i for i in range(100000)),
    sum(i * i for i in range(150000)),
    sum(i * i for i in range(200000))
]

assert results == expected, f"Expected {expected}, but got {results}"
print("Challenge 2 Passed Successfully!")

### Challenge 3: Async Rate-Limiter Context Manager

Implement an asynchronous context manager `AsyncRateLimiter` that restricts access to a block of code to `max_calls` executions per `period` seconds. If entering the context manager would exceed this limit, it must yield control and sleep using `asyncio.sleep` until enough time has passed.

In [ ]:
import asyncio
import time

class AsyncRateLimiter:
    def __init__(self, max_calls, period):
        self.max_calls = max_calls
        self.period = period
        self.timestamps = []

    async def __aenter__(self):
        # TODO: Check timestamps. Remove those older than `self.period` from the start of the list.
        # If the length of remaining timestamps is >= max_calls, sleep until the oldest timestamp falls outside the window.
        # Record the current time (after potential sleep) in `self.timestamps` and return `self`.
        pass

    async def __aexit__(self, exc_type, exc_val, exc_tb):
        # No exit action needed
        pass

In [ ]:
# Solution Code for Challenge 3
# class AsyncRateLimiter:
#     def __init__(self, max_calls, period):
#         self.max_calls = max_calls
#         self.period = period
#         self.timestamps = []
# 
#     async def __aenter__(self):
#         now = time.time()
#         self.timestamps = [t for t in self.timestamps if now - t < self.period]
#         if len(self.timestamps) >= self.max_calls:
#             wait_time = self.timestamps[0] + self.period - now
#             if wait_time > 0:
#                 await asyncio.sleep(wait_time)
#         self.timestamps.append(time.time())
#         return self
# 
#     async def __aexit__(self, exc_type, exc_val, exc_tb):
#         pass

In [ ]:
# Challenge 3 Verification Test
limiter = AsyncRateLimiter(max_calls=2, period=0.2)

async def limited_worker():
    async with limiter:
        return time.time()

t_start = time.time()
# Run 3 workers concurrently. Since limit is 2 per 0.2s, the 3rd task must be delayed.
times = await asyncio.gather(
    limited_worker(),
    limited_worker(),
    limited_worker()
)

d1 = times[0] - t_start
d2 = times[1] - t_start
d3 = times[2] - t_start

print(f"Task 1 ran after: {d1:.4f}s")
print(f"Task 2 ran after: {d2:.4f}s")
print(f"Task 3 ran after: {d3:.4f}s")

assert d3 - d1 >= 0.18, f"Task 3 should have been delayed by at least 0.18s, took {d3 - d1:.4f}s"
print("Challenge 3 Passed Successfully!")

[◀ Notebook 14: Metaprogramming](14_metaprogramming.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 16: Modules & Packages ▶](16_modules_and_packages.ipynb)